In [2]:
import numpy as np
import matplotlib.pyplot as plt
import random
from numpy.random import seed
from numpy.random import normal
import pandas as pd
import pickle
import time
import warnings
import itertools
from typing import Dict
import math
import datetime

# Turn off FutureWarning
warnings.filterwarnings("ignore")


##Interaction matrix of residents and mutants species with x trait values 
def A_res_mut(x_trait,sigma):
    diff = np.subtract.outer(x_trait, x_trait)
    #diff2 =np.subtract.outer(y_trait, y_trait)
    result = np.exp(-np.power(diff,2 )/(2*sigma**2))
    return result

#Resource competition
def r_kennel(x_trait,sigmar):
    growth = []
    for x in x_trait[0::]:
        growth.append(np.maximum(np.exp(-np.power((x),2)/(2*sigmar**2))-0.0002, 0))
    return np.array(growth)

#Variance and Expectation

def off_diag_expect_var(matrix):
    # Extract the off-diagonal entries of the matrix
    n = matrix.shape[0]
    off_diag_entries = []
    
    for i in range(n):
        for j in range(i+1, n):
            off_diag_entries.append(matrix[i][j])
    
    var_x = np.var(off_diag_entries)
    mean = np.mean(off_diag_entries)
    
    return mean, var_x


def jaccard_similarity(d: Dict[int, list], step_size: int = 5, num_keys: int = 10) -> Dict:
    result = {}
    keys = list(d.keys())[-num_keys:]
    for i in keys:
        for j in keys:
            if abs(i - j) == step_size:
                set_i = set(d[i])
                set_j = set(d[j])
                intersection = len(set_i.intersection(set_j))
                union = len(set_i.union(set_j))
                if union != 0:
                    result[(i, j)] = intersection / union
                else:
                    result[(i, j)]=0
    return result

def hill_number(abundances, q):
    p = abundances / abundances.sum()
    if q == 0:
        return len(p)
    else:
        return np.power(np.sum(np.power(p, q)), -1 / (q - 1))
    
    
def Combinations(number):
    
    # Define the possible values for each parameter
     # Generate n random floats between 0 and 1 without repeats
    param1= random.sample([random.uniform(0.5, 1) for i in range(1000)],number)#[round(random.random(),4) for _ in range(10)]
    param2= random.sample([random.uniform(0, 0.5) for i in range(1000)], number)

    # Generate n random integers without repeats
    param3=[random.uniform(0, 5) for _ in range(number)]
    ##print(len(Combinations))

    # Generate all possible combinations of values for the three parameters
    Combinations = list(itertools.product(param1, param2, param3))
    #print(combinations)
    # Print each combination
    #for combination in Combinations:
        #param1, param2, param3 = combination
        #print(f"param1: {param1}, param2: {param2}, param3: {param3}")
        #print(combination)
    return Combinations


def introduction_rate_mean(Dict,n_iterations):
    #print(len(Intro))    
    #print(cp1)
    last_ = list(Dict.values())[-n_iterations:]
    Intro_mean = sum(last_)/ len(last_)
    #Introd[params] = Intro_mean
    return Intro_mean

def RAO_(list_1, list_2):
    #RAO
    differences = []
    proportions = []

    sum_list_1 = sum(list_1)
    sum_list_2 = sum(list_2)

    for i in range(len(list_1) - 1):
        for j in range(i+1, len(list_1)):
            diff = list_1[j] - list_1[i]
            differences.append(diff**2)
            proportion_i = list_2[i] / sum_list_2
            proportion_j = list_2[j] / sum_list_2
            proportions.append(proportion_i * proportion_j)

    result = sum([diff * prop for diff, prop in zip(differences, proportions)])
    #RAO[i] = result
    return result

def RAO_mean_(Dict, n_iterations):
    #RAO_mean
    m = list(Dict.values())[-n_iterations:]
    mean= sum(m)/len(m)
    #RAO_mean[params]= mm
    return mean

def functional_turnover(Dict, lag,n_iterations):
    my_dict=Dict
    #Function turnover
    differences = []

    for i, key1 in enumerate(my_dict):
        for j in range(i + lag, len(my_dict)):
            key2 = list(my_dict.keys())[j]
            diff = my_dict[key2] - my_dict[key1]
            differences.append(abs(diff))
    if len(differences[-n_iterations:])!= 0:
        mean= sum(differences[-n_iterations:])/ len(differences[-n_iterations:])
    else:
        mean=0
    #DeltaRAO1[params] = mean
    return mean

def trait_var(Dict,n_iterations):
    t_variances = {}
    for inner_key, inner_value in Dict.items():
        if inner_value:
            mean = sum(inner_value[-n_iterations:]) / len(inner_value[-n_iterations:])
            variance = sum((x - mean) ** 2 for x in inner_value[-n_iterations:]) / len(inner_value[-n_iterations:])
            t_variances[inner_key] = variance
        else:
            t_variances[inner_key] = 0

    if len(list(t_variances.values())) != 0:
        t_variance = np.var(list(t_variances.values()), ddof=1)
    else:
        t_variance = 0
    #Trait_var[params] = t_variance
    return t_variance

def trait_mean_(Dict,n_iterations):
    t_means = {}
    for inner_key, inner_value in Dict.items():
        if inner_value:
            t_means[inner_key] = sum(inner_value[-n_iterations:]) / len(inner_value[-n_iterations:])
        else:
            t_means[inner_key] = 0

    t_mean = sum(list(t_means.values()))/len(list(t_means.values()))
    #trait_mean[params] = t_mean
    return t_mean

def mean_abundances(Dict,n_iterations):
    ssp=[]
    for k,var in Dict.items():
        ssp.append(var)
    #ssp
    # creating dataframe
    row_sum = []
    df = pd.DataFrame(ssp[-n_iterations:])
    df.fillna(0)
    ro_sum = df.sum(axis=1)
    if len(ro_sum) != 0:
        ab_mean= sum(ro_sum)/len(ro_sum)
    else:
        ab_mean = 0
    return ab_mean
def mean_inter_strength(Dict,n_iterations):
    aij_means = {}
    #print(len(aij))
    for inner_key, inner_value in Dict.items():
        #print(inner_key)
        aij_means[inner_key] = off_diag_expect_var(inner_value)[0] 
    #print('ll=',aij_means)
    if len(list(aij_means.values())[-n_iterations:])!=0:
           aij_mean = sum(list(aij_means.values())[-n_iterations:])/len(list(aij_means.values())[-n_iterations:])
    else:
           aij_mean =0
    return aij_mean
def var_inter_str(Dict,n_iterations):
    aij_variances = {}
    for inner_key, inner_value in Dict.items():
        if inner_value.any():
            mean =off_diag_expect_var(inner_value)[0]
            variance = off_diag_expect_var(inner_value)[1]
            aij_variances[inner_key] = variance
        else:
            aij_variances[inner_key] = 0
    #print('varai', aij_variances)
    if len(list(aij_variances.values())[-n_iterations:]) != 0:
        aij_variance = np.mean(list(aij_variances.values())[-n_iterations:])
    else:
        aij_variance = 0
    return aij_variance

def growth_rate_mean(Dict,n_iterations):
    ri_means = {}
    for inner_key, inner_value in Dict.items():
        if inner_value.any():
            ri_means[inner_key] = sum(inner_value[-n_iterations:]) / len(inner_value[-n_iterations:])
        else:
            ri_means[inner_key] = 0

    ri_mean = sum(list(ri_means.values()))/len(list(ri_means.values()))
    return ri_mean

def growth_rate_var(Dict,n_iterations):
    ri_variances = {}
    for inner_key, inner_value in Dict.items():
        if inner_value.any():
            mean = sum(inner_value[-n_iterations:]) / len(inner_value[-n_iterations:])
            variance = sum((x - mean) ** 2 for x in inner_value[-n_iterations:]) / len(inner_value[-n_iterations:])
            ri_variances[inner_key] = variance
        else:
            ri_variances[inner_key] = 0

    if len(list(ri_variances.values())) != 0:
        ri_variance = np.var(list(ri_variances.values()), ddof=1)
    else:
        ri_variance = 0
    return ri_variance

def Jaccard_mean(Dict, num, last_iterations):
    result = jaccard_similarity(Dict,num,last_iterations)
    merged_dict = {}
    for key, value in result.items():
        if key[::-1] in merged_dict:
            merged_dict[key[::-1]] = (merged_dict[key[::-1]] + value) / 2
        else:
            merged_dict[key] = value

    a = list(merged_dict.values())
    if len(a) != 0:
        j_mean=sum(a)/len(a)
    else:
        j_mean = 0
    return j_mean

In [2]:

def process_combination(combination):
    # start the timer
    start_time = time.time()
    param1, param2, param3 = combination


    n,start,N_iter,n_iterations =300, 200, 1800, 1000

   
    
    #species abundances
    sp_dict={}
    # structural stability limity
    sp_limit=[]
    sp_limit1=[]
    sp_limit2=[]
    #species richness
    cp1={}
    #species traits
    trait={}
    # list of species abundaces
    sp1=[]
    # sigmar 
    aij = {}
    #sigmar.append()
    ri = {}
    Intro = {}
    RAO = {}
    #initial number of species  
    
    H0 = []
    H1=[]
    H2 = []
    #nj = list(np.random.randint(5,20,200))
    current_species=list(np.arange(0,n,1))
    #print('current_species',len(current_species))
    old_n = list(np.random.randint(1500,6000,n))
    #print('old_n',len(old_n))
    xi=np.random.uniform(-1,1,n) # x trait value
    current_xi =[var for j,var in enumerate(xi) if j in current_species]
    #print('current_s',len(nJ))
    sigmar=param1
    sigma = param2
    lam = param3

    for i in range(0,start):
        if i==0:
            im=np.full((len(current_species),),0.01)
            cp1[i]=current_species
            trait[i]=current_xi
            aij[i]= A_res_mut(current_xi,sigma)
            ri[i]= r_kennel(current_xi,sigmar)
            Intro[i] = 0
            sp1.append(list(old_n))
            d = dict(zip(current_species,old_n))
            sp_dict[i] = d
            # Function for calculating expectation
            expect = off_diag_expect_var(A_res_mut(current_xi,sigma))[0]
            var = off_diag_expect_var(A_res_mut(current_xi,sigma))[1]
            #s_starv1 = np.exp(num)*(1-expect)**2/(2*var)
            s_starv2 = (1-expect)**2/(2*var)
            #sp_limit.append(s_starv1)
            sp_limit1.append(s_starv2)
            #RAO
            RAO[i] = RAO_(current_xi, old_n)

            #print(np.array(new_n))
            # Calculate Hill numbers for q = 0, 1, and 2
            h0 = hill_number(np.array(old_n), 0)
            h1 = hill_number(np.array(old_n), 1.001)
            h2 = hill_number(np.array(old_n), 2)

            H0.append(h0)
            H1.append(h1)
            H2.append(h2)
        else:
            im=np.full((len(current_species),),0.01)
            lambdas= im + old_n*np.exp((r_kennel(current_xi,sigmar)-A_res_mut(current_xi,sigma)@(old_n)*1/10000))
            #print('lam',lambdas)
            new_n= np.random.poisson(lambdas, len(current_species))
            #old_n = list(new_n)
            #print('d',d)
            cp1[i]=current_species
            trait[i]=current_xi
            aij[i]= A_res_mut(current_xi,sigma)
            ri[i]= r_kennel(current_xi,sigmar)
            Intro[i] = 0
            sp1.append(list(new_n))
            d = dict(zip(current_species,new_n))
            sp_dict[i] = d
            remove=np.array(np.where(new_n==0))
            #print(type(remove))
            keep=np.where(new_n>0)
            current_spec=[int(e) for i,e in enumerate(current_species) if i not in remove]
            #print(current_spec)
            current_species= list(np.array(current_species)[keep])
            #print('current_species',current_species)
            new_n = list(np.array(new_n)[keep])
            #print('new_n',len(new_n))
            current_xi=[var for j,var in enumerate(current_xi) if j not in remove]
            #nJ=[var for j,var in enumerate(nJ) if j not in remove]
            #print('current_xtrait',len(current_xtrait))
            old_n = list(new_n) 

            # Function for calculating expectation
            expect = off_diag_expect_var(A_res_mut(current_xi,sigma))[0]
            var = off_diag_expect_var(A_res_mut(current_xi,sigma))[1]
            #s_starv1 = np.exp(num)*(1-expect)**2/(2*var)
            s_starv2 = (1-expect)**2/(2*var)
            #sp_limit.append(s_starv1)
            sp_limit1.append(s_starv2)

            #RAO
            RAO[i] = RAO_(current_xi, new_n)

            #print(np.array(new_n))
            # Calculate Hill numbers for q = 0, 1, and 2
            h0 = hill_number(np.array(new_n), 0)
            h1 = hill_number(np.array(new_n), 1.001)
            h2 = hill_number(np.array(new_n), 2)

            H0.append(h0)
            H1.append(h1)
            H2.append(h2)



    for i in range(start,N_iter):
        im=np.full((len(current_species),),0.01)
        lambdas= im + old_n*np.exp((r_kennel(current_xi,sigmar)-A_res_mut(current_xi,sigma)@old_n*1/10000))
        ##new_n =  poisson_pmf(len(current_species),lambdas,len(idnx))
        new_n=  np.random.poisson(lambdas, len(current_species))
         #nJ =[var for j,var in enumerate(nJ) if j not in remove]
        #adding new species
        num=np.random.poisson(lam, size=None)
        #print(num)
        new_n= list(new_n) + list(np.full((num,),1000))
        current_species= current_species + random.sample(range(10000,40000),num)
        #print('current_species',len(current_species))
        current_xi=current_xi+list(np.random.uniform(-10,10,num))
        #print('current_xi',len(current_xi))
        d = dict(zip(current_species,new_n))
        sp_dict[i] = d
        cp1[i]=current_species
        trait[i]=current_xi
        aij[i]= A_res_mut(current_xi,sigma)
        ri[i]= r_kennel(current_xi,sigmar)
        Intro[i] = num
        sp1.append(list(new_n))
        remove=np.array(np.where(np.array(new_n)==0))
        #print(remove)
        keep=np.where(np.array(new_n)>0)
        #print(keep)
        current_species=[int(e) for i,e in enumerate(current_species) if i not in remove]
        #print(current_spec)
        #current_spec= list(np.array(current_spec)[keep])

        new_n = list(np.array(new_n)[keep])
        #print('new_n',len(new_n))
        current_xi=[var for j,var in enumerate(current_xi) if j not in remove]
        #print('current_xtrait',(len(current_xi)))
        #nJ =[var for j,var in enumerate(nJ) if j not in remove]

        old_n =  list(new_n) #list(new_n) + list(np.full((num,),60) )#random.sample(range(0,80),num)
        #print(len(old_n))
        #current_species= current_species + random.sample(range(10000,40000),num)
        #current_xi=current_xi+list(np.random.uniform(-1,1,num))


        # Function for calculating expectation
        mean = off_diag_expect_var(A_res_mut(current_xi,sigma))[0]
        #print('m=',mean)
        var = off_diag_expect_var(A_res_mut(current_xi,sigma))[1]
        #print('v=',var)
        s_starv1 = np.exp(num)*(1-mean)**2/(2*var)
        #print('s1=', s_starv1)
        s_starv2 = (1-mean)**2/(2*var)
        #print('s2=', s_starv2)
        sp_limit.append(s_starv1)
        sp_limit1.append(s_starv2)

        #RAO
        RAO[i] = RAO_(current_xi, new_n)

        # Calculate Hill numbers for q = 0, 1, and 2
        h0 = hill_number(np.array(new_n), 0)
        h1 = hill_number(np.array(new_n), 1.001)
        h2 = hill_number(np.array(new_n), 2)
        H0.append(h0)
        H1.append(h1)
        H2.append(h2)

    #print(len(Intro))    
    #Introduction rate
    #overall
    Richness = {}
    Shan = {}
    Shn_1 = {}
    Simp = {}
    Abund2 ={}
    Stab1 ={}
    Stab2 = {}
    Current_sp = {}
    Current_sp_var = {}
    Trait_var ={}
    trait_mean ={}
    Aij ={}
    Aij_var={}
    Ri ={}
    Ri_var={}
    Introd={}
    Jaccard_lag1={}
    Jaccard_lag5={}
    Jaccard_lag10={}
    Jaccard_lag50={}
    Jaccard_lag100={}
    Jaccard_lag1000={}
    DeltaRAO1={}
    DeltaRAO5={}
    DeltaRAO10={}
    DeltaRAO50={}
    DeltaRAO100={}
    DeltaRAO1000={}
    RAO_mean ={}
    Introd[combination] = introduction_rate_mean(Intro,n_iterations)

    #RAO_mean
    RAO_mean[combination]= RAO_mean_(RAO,n_iterations)

    #Function turnover
    DeltaRAO1[combination] = functional_turnover(RAO, 1,n_iterations)
    DeltaRAO5[combination] = functional_turnover(RAO, 5,n_iterations)
    DeltaRAO10[combination] = functional_turnover(RAO, 10,n_iterations)
    DeltaRAO50[combination] = functional_turnover(RAO, 50,n_iterations)
    DeltaRAO100[combination] = functional_turnover(RAO, 100,n_iterations)
    DeltaRAO1000[combination] = functional_turnover(RAO, 200,n_iterations)

    rich_var_count = {inner_key: len(inner_value) for inner_key, inner_value in cp1.items()}
    rich_var_count_values = list(rich_var_count.values())[-n_iterations:]
    rmean = sum(rich_var_count_values) / len(rich_var_count_values) if len(rich_var_count_values) != 0 else 0
    variance = sum((x - rmean) ** 2 for x in rich_var_count_values) / len(rich_var_count_values) if len(rich_var_count_values) != 0 else 0
    Current_sp_var[combination] = variance

    #Richness
    rich_count = {inner_key: len(inner_value) for inner_key, inner_value in cp1.items()}
    last_counts = list(rich_count.values())[-n_iterations:]
    rmean = sum(last_counts) / len(last_counts) if len(last_counts) != 0 else 0
    Current_sp[combination] = rmean

    #Mean Shannon Entropy
    H_shan = [math.log(x) for x in H1]
    sha_mean = sum(H_shan[-n_iterations:]) / len(H_shan[-n_iterations:]) if len(H_shan[-n_iterations:]) != 0 else 0 
    Shan[combination] = sha_mean

    #Mean Simpson Entropy
    Simpson=np.power(H2,-1)
    if len(Simpson[-n_iterations:])!= 0:
        shi_mean = sum(Simpson[-n_iterations:])/len(Simpson[-n_iterations:])
    else:
        shi_mean = 0 
    Simp[combination] =shi_mean

    # S_star with Introduction rate
    if len(sp_limit[-n_iterations:]) != 0:
        st_mean = sum(sp_limit[-n_iterations:])/len(sp_limit[-n_iterations:])
    else:
        st_mean = 0
    Stab1[combination] = st_mean

    #S_star 
    if len(sp_limit1[-n_iterations:])!=0:
        st1_mean = sum(sp_limit1[-n_iterations:])/len(sp_limit1[-n_iterations:])
    else:
        st1_mean =0
    Stab2[combination] = st1_mean

    #variance of traits
    Trait_var[combination] = trait_var(trait,n_iterations)

    # mean of traits
    trait_mean[combination] = trait_mean_(trait,n_iterations)

    #mean of abundances
    Abund2[combination] = mean_abundances(sp_dict,n_iterations)

    # Calculate the mean of the arrays in each inner dict
    Aij[combination] = mean_inter_strength(aij,n_iterations)

    #variance of interaction strength
    Aij_var[combination] = var_inter_str(aij,n_iterations)

    #Intrinsic growth rate
    Ri[combination] = growth_rate_mean(ri,n_iterations)

    Ri_var[combination] = growth_rate_var(ri,n_iterations)

    #Jaccard_mean
    Jaccard_lag1[combination] = Jaccard_mean(cp1, 1, n_iterations)
    Jaccard_lag5[combination] = Jaccard_mean(cp1, 5, n_iterations)
    Jaccard_lag10[combination] = Jaccard_mean(cp1, 10, n_iterations)
    Jaccard_lag50[combination] = Jaccard_mean(cp1, 50, n_iterations)
    Jaccard_lag100[combination] = Jaccard_mean(cp1, 100, n_iterations)
    Jaccard_lag1000[combination] = Jaccard_mean(cp1, 200, n_iterations)
    
    
    return {combination:{
        #'Richness': Richness.values(),
        'Shan': float(Shan.values().__iter__().__next__()),
        #'Shn_1': Shn_1.values().__iter__().__next__()),
        'Simp': float(Simp.values().__iter__().__next__()),
        'Abund2': float(Abund2.values().__iter__().__next__()),
        'Stab1': float(Stab1.values().__iter__().__next__()),
        'Stab2': float(Stab2.values().__iter__().__next__()),
        'Current_sp': float(Current_sp.values().__iter__().__next__()),
        'Current_sp_var': float(Current_sp_var.values().__iter__().__next__()),
        'Trait_var': float(Trait_var.values().__iter__().__next__()),
        'trait_mean': float(trait_mean.values().__iter__().__next__()),
        'Aij': float(Aij.values().__iter__().__next__()),
        'Aij_var': float(Aij_var.values().__iter__().__next__()),
        'Ri': float(Ri.values().__iter__().__next__()),
        'Ri_var': float(Ri_var.values().__iter__().__next__()),
        'Introd': float(Introd.values().__iter__().__next__()),
        'Jaccard_lag1': float(Jaccard_lag1.values().__iter__().__next__()),
        'Jaccard_lag5': float(Jaccard_lag5.values().__iter__().__next__()),
        'Jaccard_lag10': float(Jaccard_lag10.values().__iter__().__next__()),
        'Jaccard_lag50': float(Jaccard_lag50.values().__iter__().__next__()),
        'Jaccard_lag100': float(Jaccard_lag100.values().__iter__().__next__()),
        'Jaccard_lag1000': float(Jaccard_lag200.values().__iter__().__next__()),
        'DeltaRAO1': float(DeltaRAO1.values().__iter__().__next__()),
        'DeltaRAO5': float(DeltaRAO5.values().__iter__().__next__()),
        'DeltaRAO10': float(DeltaRAO10.values().__iter__().__next__()),
        'DeltaRAO50': float(DeltaRAO50.values().__iter__().__next__()),
        'DeltaRAO100': float(DeltaRAO100.values().__iter__().__next__()),
        'DeltaRAO1000': float(DeltaRAO200.values().__iter__().__next__()),
        'RAO_mean': float(RAO_mean.values().__iter__().__next__())}
    }
    
    
 

In [3]:
from joblib import Parallel, delayed
from tqdm import tqdm
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# start the timer
start_time = time.time()
number =10
if __name__ == '__main__':
    
    with open('Combinations_hpc', 'rb') as f:
        combinations=pickle.load(f)
        
    print("Processing started...")

    results = Parallel(n_jobs=48)(delayed(process_combination)(combination) for combination in tqdm(combinations))

    print("Processing completed.")

# Combine the results into a single dictionary
results_dict = {}
for result in results:
    results_dict.update(result)
#file_name = f'comb_{current_time}'
#with open(file_name, 'wb') as f:
   # pickle.dump(combinations,f)
    
file_name = f'result_{current_time}'  
with open(file_name, 'wb') as f:
    pickle.dump(results_dict,f)
# stop the timer
end_time = time.time()

# calculate the elapsed time
elapsed_time = end_time - start_time

# print the elapsed time in seconds
print("Elapsed time:", elapsed_time, "seconds")

Processing started...


100%|████████████████████████████████████████████████████████████████████████████| 1000/1000 [3:01:30<00:00, 10.89s/it]


Processing completed.
Elapsed time: 12491.605348348618 seconds


In [6]:
with open('result_2023-05-22_16-20-58', 'rb') as f:
        res=pickle.load(f)

In [7]:
B= list(res.keys())
# create dataframe with columns named "Column1", "Column2", "Column3"
df_ex1 = pd.DataFrame(B, columns=["sigmar", "sigma", "lamda"])
df_ex1

,sigmar,sigma,lamda
0,0.536935,0.29414,0.742032
1,0.536935,0.29414,2.484732
2,0.536935,0.29414,0.341722
3,0.536935,0.29414,0.305252
4,0.536935,0.29414,3.365534
...,...,...,...
995,0.577255,0.15826,1.566830
996,0.577255,0.15826,1.533918
997,0.577255,0.15826,3.362893
998,0.577255,0.15826,0.498928


In [8]:
dfs = []
for key, value in res.items():
    #print(value)
    df = pd.DataFrame([value])
    #print(df)
    dfs.append(df)

combined_df_1 = pd.concat(dfs, ignore_index=True)

In [9]:
combs= pd.concat( [df_ex1,combined_df_1],  ignore_index=True)